# 05 · Bonds: Fixed, Floating, and Hybrids

Finstack ships multiple bond constructors: Treasury-style helpers, fully parameterized builders, floating-rate notes, amortizing loans, and schedule-driven hybrids. This notebook highlights the patterns used in exotic credit and private credit desks.

### Learning Objectives
- Build liquid discount/forward markets once and reuse them across bond variants
- Instantiate fixed, floating, zero-coupon, and Treasury bonds with helper constructors
- Feed custom cashflow schedules (PIK/amortizing) into `Bond.from_cashflows`
- Compute asset-swap spreads and analytics such as clean price, yield, Z-spread, and DV01

In [ ]:
from datetime import date

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve, ForwardCurve
from finstack.valuations.cashflow import (
    CashflowBuilder,
    ScheduleParams,
    FixedCouponSpec,
    CouponType,
    AmortizationSpec,
)
from finstack.valuations.instruments import Bond
from finstack.valuations.metrics import MetricId
from finstack.valuations.pricer import create_standard_registry

as_of = date(2025, 1, 2)

def build_market(as_of: date) -> MarketContext:
    market = MarketContext()
    usd_ois = DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9975), (1.0, 0.9940), (3.0, 0.9725), (5.0, 0.9440)],
    )
    treasury = DiscountCurve(
        "USD-TREASURY",
        as_of,
        [(0.0, 1.0), (5.0, 0.9750), (10.0, 0.9300)],
    )
    sofr_3m = ForwardCurve(
        "USD-SOFR-3M",
        0.25,
        [(0.0, 0.045), (1.0, 0.0465), (3.0, 0.0480), (5.0, 0.0495)],
        base_date=as_of,
    )
    market.insert_discount(usd_ois)
    market.insert_discount(treasury)
    market.insert_forward(sofr_3m)
    return market

market = build_market(as_of)
registry = create_standard_registry()
notional = Money(10_000_000, USD)


## 1. Core Constructors
The bindings expose ergonomic helpers for common structures. We value a Treasury-style bond, a standard corporate fixed-rate note, a floating-rate note, and a zero-coupon bond.

In [ ]:
treasury = Bond.treasury(
    "UST-5Y",
    notional,
    0.03625,
    issue=date(2023, 11, 15),
    maturity=date(2028, 11, 15),
)
corp_fixed = Bond.fixed_semiannual(
    "ACME-31",
    notional,
    0.045,
    issue=as_of,
    maturity=date(2030, 1, 2),
    discount_curve="USD-OIS",
)
frn = Bond.floating(
    "ACME-FRN",
    notional,
    issue=as_of,
    maturity=date(2029, 1, 2),
    discount_curve="USD-OIS",
    forward_curve="USD-SOFR-3M",
    margin_bp=125.0,
)
zcb = Bond.zero_coupon(
    "ACME-ZCB",
    notional,
    issue=as_of,
    maturity=date(2030, 1, 2),
    discount_curve="USD-OIS",
)

for instrument in (treasury, corp_fixed, frn, zcb):
    res = registry.price(instrument, "discounting", market)
    print(f"{instrument.instrument_id:<12} PV = {res.value.amount:,.2f} {res.value.currency}")


## 2. Cashflow Builder Integration
For bespoke structures (PIK toggles, amortization, callable schedules) create a schedule via `CashflowBuilder` and wrap it with `Bond.from_cashflows`.

In [ ]:
builder = CashflowBuilder.new()
builder.principal(
    amount=notional.amount,
    currency=USD,
    issue=as_of,
    maturity=date(2032, 1, 2),
)
# Semi-annual 5% coupon with 60% cash / 40% PIK split
builder.fixed_cf(
    FixedCouponSpec.new(
        rate=0.05,
        schedule=ScheduleParams.semiannual_30360(),
        coupon_type=CouponType.split(0.60, 0.40),
    )
)
# Linear amortization to 70% of original notional
builder.amortization(AmortizationSpec.linear_to(Money(7_000_000, USD)))
custom_schedule = builder.build()
print("First five flows (date, amount, kind):")
for flow in custom_schedule.flows()[:5]:
    dt, amt, kind, accr, reset = flow.to_tuple()
    print(f"  {dt} {str(kind):<10} {amt.amount:,.2f} {amt.currency}")

custom_bond = Bond.from_cashflows(
    instrument_id="ACME-CUSTOM",
    schedule=custom_schedule,
    discount_curve="USD-OIS",
    quoted_clean=99.40,
)
print("Custom schedule PV:", registry.price(custom_bond, "discounting", market).value)


## 3. Asset-Swap Spreads
Given a floating reference curve and quoted dirty price, the registry can compute par and market asset-swap spreads. This is useful for relative-value screens.

In [ ]:
par_asw, mkt_asw = registry.asw_forward(
    corp_fixed,
    market,
    forward_curve="USD-SOFR-3M",
    float_margin_bp=110.0,
)
print(f"Par ASW:  {par_asw:.2f} bp")
print(f"Mkt ASW: {mkt_asw:.2f} bp")


## 4. Bond Analytics
Combine helper metrics to capture pricing, yield, spread, and duration packages in a single call.

In [ ]:
metrics = [
    "clean_price",
    "dirty_price",
    "accrued",
    "ytm",
    "duration_mod",
    "duration_mac",
    "dv01",
    "convexity",
    "z_spread",
    "i_spread",
]
res = registry.price_with_metrics(corp_fixed, "discounting", market, metrics)
for name in metrics:
    print(f"{name:>12}: {res.measures.get(name)}")


## Summary
- Helper constructors (`treasury`, `fixed_semiannual`, `floating`, `zero_coupon`) cover the most common issuance styles.
- `CashflowBuilder` unlocks bespoke amortizing/PIK programs that can be promoted into bonds for pricing and risk.
- Asset-swap utilities and metric bundles surface trader-ready analytics (clean price, yield, spreads, DV01) without hand-written ladders.